# CAFA-6 Infer Submodels From Saved Artifacts

Notebook này **không train lại**. Nó load folder artifact đã lưu từ notebook training:

```text
cafa6_high_performance_artifacts/
```

và xuất prediction TSV riêng cho từng model con trên held-out test split:

- `esm_mlp.tsv`
- `protcnn.tsv`
- `bilstm_attention.tsv`
- `protbert_mlp.tsv` nếu có
- `ensemble.tsv`

Output được ghi vào folder mới để tải về hoặc chạy `CAFA-evaluator-PK` so sánh các model.

In [ ]:
# =============================
# 0. Configuration and path discovery
# =============================
from pathlib import Path
import json
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[CONFIG] DEVICE={DEVICE}")

# If running on Kaggle, attach the artifact folder as an input dataset.
# The auto-discovery below searches common paths.
ARTIFACT_CANDIDATES = [
    Path("/kaggle/input/notebooks/qundngdngminhqun/cafa6/cafa6_high_performance_artifacts")
    Path("/kaggle/input/cafa6-high-performance-artifacts/cafa6_high_performance_artifacts"),
    Path("/kaggle/input/cafa6-offline-realtime-solution/cafa6_high_performance_artifacts"),
    Path("/kaggle/input/notebooks/qundngdngminhqun/cafa6/cafa6_high_performance_artifacts"),
    Path("/kaggle/input/notebooks/qundngdngminhqun/cafa6-offline-realtime-solution/cafa6_high_performance_artifacts"),
    Path("/kaggle/working/cafa6_high_performance_artifacts"),
    Path("cafa6_high_performance_artifacts"),
]

def find_artifact_dir():
    for p in ARTIFACT_CANDIDATES:
        if (p / "config.json").exists() and ((p / "branch_checkpoints").exists() or (p / "cafa6_high_performance_models.pt").exists()):
            return p
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("cafa6_high_performance_artifacts"):
            if (p / "config.json").exists():
                return p
    raise FileNotFoundError("Cannot find cafa6_high_performance_artifacts. Attach it as Kaggle input or set ARTIFACT_DIR manually.")

ARTIFACT_DIR = find_artifact_dir()
OUTPUT_DIR = Path("/kaggle/working/cafa6_submodel_predictions") if Path("/kaggle/working").exists() else Path("cafa6_submodel_predictions")
PRED_DIR = OUTPUT_DIR / "predictions"
PRED_VALID_DIR = OUTPUT_DIR / "predictions_valid"
for p in [OUTPUT_DIR, PRED_DIR, PRED_VALID_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"[PATH] ARTIFACT_DIR={ARTIFACT_DIR}")
print(f"[PATH] OUTPUT_DIR={OUTPUT_DIR}")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print("  -", p.name)

In [ ]:
# =============================
# 1. Load config, metadata and split IDs
# =============================
cfg = json.loads((ARTIFACT_DIR / "config.json").read_text())
selected_terms = json.loads((ARTIFACT_DIR / "go_terms.json").read_text())
go_meta = json.loads((ARTIFACT_DIR / "go_metadata.json").read_text())

split_dir = ARTIFACT_DIR / "splits"
official_eval_dir = ARTIFACT_DIR / "official_eval"

def read_ids(path):
    return [line.strip().split()[0] for line in Path(path).read_text().splitlines() if line.strip()]

test_ids = read_ids(split_dir / "test_ids.txt" if (split_dir / "test_ids.txt").exists() else official_eval_dir / "test_ids.txt")
valid_ids = read_ids(split_dir / "valid_ids.txt" if (split_dir / "valid_ids.txt").exists() else official_eval_dir / "valid_ids.txt")

print(f"[LOAD] selected_terms={len(selected_terms):,}")
print(f"[LOAD] valid_ids={len(valid_ids):,}")
print(f"[LOAD] test_ids={len(test_ids):,}")
print(json.dumps({k: cfg[k] for k in ['max_labels','embedding_model_name','embedding_max_length','sequence_max_length','official_eval_top_k'] if k in cfg}, indent=2))

# Copy evaluator support files into new output folder.
for name in ["ground_truth.tsv", "valid_ground_truth.tsv", "terms_of_interest.tsv", "test_ids.txt", "valid_ids.txt", "train_ids.txt"]:
    src = official_eval_dir / name
    if src.exists():
        dst = OUTPUT_DIR / name
        dst.write_bytes(src.read_bytes())
        print(f"[COPY] {src} -> {dst}")

In [ ]:
# =============================
# 2. Model definitions matching training notebook
# =============================
AA = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_idx = {aa: i + 1 for i, aa in enumerate(AA)}

class EmbeddingMLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dims=(1024, 512), dropout=0.35):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class ProtCNN(nn.Module):
    def __init__(self, output_dim, vocab_size=21, emb_dim=128, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.conv3 = nn.Sequential(nn.Conv1d(emb_dim, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU())
        self.conv5 = nn.Sequential(nn.Conv1d(emb_dim, 256, 5, padding=2), nn.BatchNorm1d(256), nn.ReLU())
        self.conv7 = nn.Sequential(nn.Conv1d(emb_dim, 256, 7, padding=3), nn.BatchNorm1d(256), nn.ReLU())
        self.conv = nn.Sequential(nn.Conv1d(768, 512, 3, padding=1), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout))
        self.head = nn.Sequential(nn.Linear(1024, 1024), nn.ReLU(), nn.Dropout(dropout), nn.Linear(1024, output_dim))
    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = torch.cat([self.conv3(x), self.conv5(x), self.conv7(x)], dim=1)
        x = self.conv(x)
        gap = torch.mean(x, dim=2)
        gmp = torch.max(x, dim=2).values
        return self.head(torch.cat([gap, gmp], dim=1))

class BiLSTMAttention(nn.Module):
    def __init__(self, output_dim, vocab_size=21, emb_dim=128, hidden=256, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm1 = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.MultiheadAttention(hidden * 2, num_heads=8, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden * 2)
        self.lstm2 = nn.LSTM(hidden * 2, hidden // 2, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 512), nn.ReLU(), nn.Dropout(dropout), nn.Linear(512, output_dim))
    def forward(self, x):
        pad_mask = x.eq(0)
        x = self.embedding(x)
        x, _ = self.lstm1(x)
        attn_out, _ = self.attn(x, x, x, key_padding_mask=pad_mask)
        x = self.norm(x + self.dropout(attn_out))
        x, _ = self.lstm2(x)
        valid = (~pad_mask).unsqueeze(-1).to(x.dtype)
        gap = (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
        masked = x.masked_fill(pad_mask.unsqueeze(-1), -1e4)
        gmp = masked.max(dim=1).values
        return self.head(torch.cat([gap, gmp], dim=1))

print("[MODEL] definitions ready")

In [ ]:
# =============================
# 3. Load branch checkpoints
# =============================
def load_branch_states():
    states = {}
    ckpt_dir = ARTIFACT_DIR / "branch_checkpoints"
    if ckpt_dir.exists():
        for p in sorted(ckpt_dir.glob("*.pt")):
            obj = torch.load(p, map_location="cpu")
            branch = obj.get("branch", p.stem)
            states[branch] = obj.get("state_dict", obj)
            print(f"[CKPT] loaded branch {branch}: {p.name}")
    full_ckpt = ARTIFACT_DIR / "cafa6_high_performance_models.pt"
    if full_ckpt.exists():
        obj = torch.load(full_ckpt, map_location="cpu")
        for branch, state in obj.get("model_states", {}).items():
            states.setdefault(branch, state)
            print(f"[CKPT] available in full checkpoint: {branch}")
    return states

model_states = load_branch_states()
print("[CKPT] branches:", sorted(model_states))
if not model_states:
    raise FileNotFoundError("No branch checkpoint or full model checkpoint found.")

In [ ]:
# =============================
# 4. Load embeddings and rebuild sequence tensors
# =============================
def embedding_filename(split_name, model_name, max_length):
    safe_model = model_name.replace('/', '__')
    return f"{split_name}_{safe_model}_labels{cfg['max_labels']}_len{max_length}.npy"

def find_embedding(split_name, model_name, max_length):
    filename = embedding_filename(split_name, model_name, max_length)
    candidates = [ARTIFACT_DIR / "embeddings" / filename]
    if Path("/kaggle/input").exists():
        candidates.extend(Path("/kaggle/input").rglob(filename))
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Missing embedding cache: {filename}")

Xesm_test = Xesm_valid = None
if "esm_mlp" in model_states:
    Xesm_test = np.load(find_embedding("test", cfg["embedding_model_name"], cfg["embedding_max_length"]))
    Xesm_valid = np.load(find_embedding("valid", cfg["embedding_model_name"], cfg["embedding_max_length"]))
    print(f"[EMB] Xesm_test={Xesm_test.shape}, Xesm_valid={Xesm_valid.shape}")

# Sequence models need original sequences. Use CAFA train FASTA and split ids.
def discover_cafa_base_dir():
    candidates = [
        Path('/kaggle/input/cafa-6-protein-function-prediction'),
        Path('/kaggle/input/cafa6'),
        Path.cwd(),
        Path.cwd() / 'cafa6',
    ]
    for base in candidates:
        if (base / 'Train' / 'train_sequences.fasta').exists():
            return base
    if Path('/kaggle/input').exists():
        for p in Path('/kaggle/input').rglob('train_sequences.fasta'):
            if p.parent.name == 'Train':
                return p.parent.parent
    raise FileNotFoundError('Cannot find CAFA train_sequences.fasta')

def extract_uniprot_id(header_line):
    token = header_line.strip()[1:].split()[0] if header_line.startswith('>') else header_line.strip().split()[0]
    if '|' in token:
        parts = token.split('|')
        if len(parts) >= 2 and parts[1]:
            return parts[1]
    return token

def load_fasta(filepath):
    sequences = {}
    current_id = None
    current_seq = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_id is not None:
                    sequences[current_id] = ''.join(current_seq)
                current_id = extract_uniprot_id(line)
                current_seq = []
            else:
                current_seq.append(line.upper())
        if current_id is not None:
            sequences[current_id] = ''.join(current_seq)
    return sequences

def encode_sequence(seq, max_len):
    arr = np.zeros(max_len, dtype=np.int64)
    for i, aa in enumerate(seq[:max_len]):
        arr[i] = aa_to_idx.get(aa, 0)
    return arr

def build_sequence_matrix(ids, sequences):
    X = np.zeros((len(ids), cfg['sequence_max_length']), dtype=np.int64)
    for i, pid in enumerate(ids):
        X[i] = encode_sequence(sequences[pid], cfg['sequence_max_length'])
    return X

need_sequences = any(branch in model_states for branch in ["protcnn", "bilstm_attention"])
if need_sequences:
    base_dir = discover_cafa_base_dir()
    sequences = load_fasta(base_dir / 'Train' / 'train_sequences.fasta')
    Xseq_test = build_sequence_matrix(test_ids, sequences)
    Xseq_valid = build_sequence_matrix(valid_ids, sequences)
    print(f"[SEQ] Xseq_test={Xseq_test.shape}, Xseq_valid={Xseq_valid.shape}")
else:
    Xseq_test = Xseq_valid = None

In [ ]:
# =============================
# 5. Predict probabilities for each branch
# =============================
@torch.no_grad()
def predict_numpy(model, X, batch_size):
    model = model.to(DEVICE).eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(X)), batch_size=batch_size, shuffle=False, num_workers=0)
    chunks = []
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        chunks.append(torch.sigmoid(model(xb)).cpu().numpy().astype(np.float32))
    return np.vstack(chunks)

preds_test = {}
preds_valid = {}
output_dim = len(selected_terms)

if "esm_mlp" in model_states:
    model = EmbeddingMLP(Xesm_test.shape[1], output_dim, cfg['esm_hidden_dims'], cfg['esm_dropout'])
    model.load_state_dict(model_states['esm_mlp'])
    preds_test['esm_mlp'] = predict_numpy(model, Xesm_test, cfg.get('esm_batch_size', 512))
    preds_valid['esm_mlp'] = predict_numpy(model, Xesm_valid, cfg.get('esm_batch_size', 512))
    print("[PRED] esm_mlp", preds_test['esm_mlp'].shape)
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()

if "protcnn" in model_states:
    model = ProtCNN(output_dim, dropout=cfg['protcnn_dropout'])
    model.load_state_dict(model_states['protcnn'])
    preds_test['protcnn'] = predict_numpy(model, Xseq_test, cfg.get('sequence_batch_size', 256))
    preds_valid['protcnn'] = predict_numpy(model, Xseq_valid, cfg.get('sequence_batch_size', 256))
    print("[PRED] protcnn", preds_test['protcnn'].shape)
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()

if "bilstm_attention" in model_states:
    model = BiLSTMAttention(output_dim, dropout=cfg['bilstm_dropout'])
    model.load_state_dict(model_states['bilstm_attention'])
    bs = max(32, cfg.get('sequence_batch_size', 256) // 4)
    preds_test['bilstm_attention'] = predict_numpy(model, Xseq_test, bs)
    preds_valid['bilstm_attention'] = predict_numpy(model, Xseq_valid, bs)
    print("[PRED] bilstm_attention", preds_test['bilstm_attention'].shape)
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()

if "protbert_mlp" in model_states:
    print("[WARN] protbert_mlp checkpoint exists, but this notebook does not recompute ProtBERT embeddings by default.")
    print("[WARN] Add ProtBERT embedding extraction if you trained this branch and need its TSV.")

# Ensemble from available branches.
active_weights = cfg.get('active_ensemble_weights') or {k: cfg['ensemble_weights'].get(k, 1.0) for k in preds_test}
active_weights = {k: active_weights.get(k, 0.0) for k in preds_test}
weight_sum = sum(active_weights.values())
active_weights = {k: v / weight_sum for k, v in active_weights.items()}
preds_test['ensemble'] = sum(active_weights[k] * preds_test[k] for k in active_weights)
preds_valid['ensemble'] = sum(active_weights[k] * preds_valid[k] for k in active_weights)
print("[PRED] ensemble weights", active_weights)
print("[PRED] branches ready:", sorted(preds_test))

In [ ]:
# =============================
# 6. Write prediction TSV files for CAFA-evaluator-PK
# =============================
def write_prediction_tsv(ids, probs, output_path, top_k):
    rows = []
    for i, pid in enumerate(ids):
        order = np.argsort(-probs[i])[:top_k]
        for j in order:
            score = float(probs[i, j])
            if score > 0.0:
                rows.append((pid, selected_terms[j], min(score, 1.0)))
    pd.DataFrame(rows).to_csv(output_path, sep='\t', header=False, index=False)
    print(f"[WRITE] {output_path} rows={len(rows):,}")
    return len(rows)

top_k = int(cfg.get('official_eval_top_k', 1500))
for name, probs in preds_test.items():
    write_prediction_tsv(test_ids, probs, PRED_DIR / f"{name}.tsv", top_k)
for name, probs in preds_valid.items():
    write_prediction_tsv(valid_ids, probs, PRED_VALID_DIR / f"{name}.tsv", top_k)

# Also save compact numpy probabilities for fast local analysis if needed.
for name, probs in preds_test.items():
    np.save(OUTPUT_DIR / f"test_{name}_probabilities.npy", probs.astype(np.float32))
for name, probs in preds_valid.items():
    np.save(OUTPUT_DIR / f"valid_{name}_probabilities.npy", probs.astype(np.float32))

print("[DONE] Prediction folder for held-out test:", PRED_DIR)
print("[DONE] Prediction folder for validation:", PRED_VALID_DIR)
print("[DONE] Use these folders as pred_dir in CAFA-evaluator-PK.")

In [ ]:
# =============================
# 7. Example evaluator commands
# =============================
print("Test split evaluator command:")
print(f"""
python /kaggle/working/CAFA-evaluator-PK/src/cafaeval/__main__.py \\
  /kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo \\
  {PRED_DIR} \\
  {OUTPUT_DIR / 'ground_truth.tsv'} \\
  -out_dir {OUTPUT_DIR / 'results_test'} \\
  -ia /kaggle/input/cafa-6-protein-function-prediction/IA.tsv \\
  -toi {OUTPUT_DIR / 'terms_of_interest.tsv'} \\
  -th_step 0.01 \\
  -threads 4
""")
print("Validation split evaluator command:")
print(f"""
python /kaggle/working/CAFA-evaluator-PK/src/cafaeval/__main__.py \\
  /kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo \\
  {PRED_VALID_DIR} \\
  {OUTPUT_DIR / 'valid_ground_truth.tsv'} \\
  -out_dir {OUTPUT_DIR / 'results_valid'} \\
  -ia /kaggle/input/cafa-6-protein-function-prediction/IA.tsv \\
  -toi {OUTPUT_DIR / 'terms_of_interest.tsv'} \\
  -th_step 0.01 \\
  -threads 4
""")